<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">MODULE 9: WAREHOUSE MODELING AND JOINS</div><div style="color:#17212b;font-size:30px;font-weight:750">Move from a single imported order table to a small fact-and-dimension model</div><p style="color:#475569;line-height:1.7">Run in order against a dedicated Level 1 course database. Every result is rendered as a table and every write is scoped to this module's objects.</p></div>

## Boundary

This lab never changes the Level 1 source tables. It creates or replaces only objects with the `_l2` suffix.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_fact_l2")
lab.execute("DROP TABLE IF EXISTS customers_dim_l2")
lab.execute("""
CREATE TABLE customers_dim_l2 (
    customer_id BIGINT NOT NULL,
    customer_name VARCHAR(100) NOT NULL,
    region VARCHAR(32) NOT NULL
)
UNIQUE KEY(customer_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1", "enable_unique_key_merge_on_write"="true")
""")
lab.execute("""
CREATE TABLE orders_fact_l2 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.execute("INSERT INTO customers_dim_l2 SELECT customer_id, customer_name, 'known' FROM customers LIMIT 20")
lab.execute("INSERT INTO orders_fact_l2 SELECT order_date, order_id, customer_id, order_amount, data_source FROM orders_imported")
lab.sql("SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS order_grain FROM orders_fact_l2", title="Fact grain contract")

In [ ]:
lab.sql("""
SELECT f.order_id, f.order_amount, d.customer_name, d.region
FROM orders_fact_l2 AS f
LEFT JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
ORDER BY f.order_id
LIMIT 10
""", title="Fact and customer dimension")
lab.sql("""
SELECT f.customer_id, COUNT(*) AS orders_without_dimension
FROM orders_fact_l2 AS f
LEFT JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
WHERE d.customer_id IS NULL
GROUP BY f.customer_id
ORDER BY f.customer_id
""", title="Anti-join check for missing customers")

In [ ]:
lab.sql("""
SELECT f.region, SUM(f.order_amount) AS amount
FROM orders_fact_l2 AS f
JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
GROUP BY f.region
ORDER BY f.region
""", title="Dimension-enriched metric")
lab.sql("""
EXPLAIN
SELECT f.order_id, d.customer_name
FROM orders_fact_l2 AS f
JOIN customers_dim_l2 AS d ON d.customer_id = f.customer_id
""", title="Join distribution plan", final=True)

## Takeaway

Compare the result with the business grain stated in the lesson. A successful SQL statement is not by itself evidence that the model, metric, access boundary, or consumer contract is correct.